In [3]:
import joblib
import numpy as np
import pandas as pd

BUNDLE_PATH = "stack_lgbm_iso_bundle.joblib"

In [4]:
categorical_feats = [
    'Location_Region', 'VRLOCATION', 'Trim', 'Series', 'Vehicle_botcolor', 'Vehicle_trantype', 'Vehicle_drive', 
    'Vehicle_condition_drivable', 'BodyClass', 'DriveType', 'FuelTypePrimary', 'EngineConfiguration', 'EngineModel', 
    'GVWR', 'PlantCountry', 'IsEV', 'EVType', 'MarketSegment', 'Model', 'Make','Misc_SalesChannel'
]
# …and add it into numeric_feats
numeric_feats = [
    'VRMILEAGE','Vehicle_year','Vehicle_cylinders','Vehicle_doors','Vehicle_engine',
    'Vehicle_condition_overall','EngineHP','BasePrice','SALEDATE_WeekofYearNumber','SALEDATE_MonthofYearNumber',
    'SALEDATE_Quarter','SALEDATE_Year','vehicle_age','mileage_per_year','log_mileage','log_age',
    'drivable_flag','GVWR_class','Vehicle_condition_overall_missing','condition_imputed_from_lights','EngineHP_missing','BasePrice_missing','Vehicle_cylinders_missing'
]

In [5]:
import numpy as np
import pandas as pd

def clean_and_fix_temporal(df, k_smooth=100, drop_vrlights=True, drop_dates=True):
    """
    Cleaning + feature fixes.
    - Robustly parse VRSALEDATE into sale_date (handles int/float/datetime/strings)
    - Derive/repair SALEDATE_* components from sale_date (universal, not last-N-days)
    - Fill missing Vehicle_condition_overall using VRLIGHTS (universal smoothed-mean mapping)
    - Keep final column name Vehicle_condition_overall (fill in-place)
    - Smarter categorical missing handling:
        * Use "Not Applicable" only when concept truly doesn't apply (mostly EV vs ICE logic)
        * Otherwise use "Unknown"
    - Optionally drop VRSALEDATE and sale_date at end (default: True)
    """

    df = df.copy()

    # ------------------------------------------------------------------
    # 0) Column name hygiene (prevents "VRSALEDATE " trailing-space issues)
    # ------------------------------------------------------------------
    df.columns = df.columns.astype(str).str.strip()

    # ------------------------------------------------------------------
    # 1) Parse sale date + basic temporal fields (ROBUST)
    # ------------------------------------------------------------------
    df["sale_date"] = pd.NaT
    if "VRSALEDATE" in df.columns:
        s = df["VRSALEDATE"]

        # Case A: already datetime-like (common when pulling from DB)
        if np.issubdtype(s.dtype, np.datetime64):
            df["sale_date"] = s
        else:
            # Convert to clean strings
            s2 = (
                s.astype("string")
                 .str.strip()
                 # floats like 20250131.0 -> "20250131"
                 .str.replace(r"\.0$", "", regex=True)
            )

            # Strict 8-digit YYYYMMDD mask
            digits8 = s2.str.fullmatch(r"\d{8}")

            # Parse YYYYMMDD when applicable
            sale_date_ymd = pd.to_datetime(
                s2.where(digits8),
                format="%Y%m%d",
                errors="coerce"
            )

            # Parse everything else by inference (YYYY-MM-DD etc.)
            sale_date_infer = pd.to_datetime(
                s2.where(~digits8),
                errors="coerce"
            )

            df["sale_date"] = sale_date_ymd.fillna(sale_date_infer)

    # Ensure SALEDATE_ fields numeric (if present)
    for c in ["SALEDATE_Year", "SALEDATE_MonthofYearNumber", "SALEDATE_WeekofYearNumber", "SALEDATE_Quarter"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    # Derive missing SALEDATE components from sale_date
    if df["sale_date"].notna().any():
        if "SALEDATE_Year" not in df.columns:
            df["SALEDATE_Year"] = df["sale_date"].dt.year
        else:
            df["SALEDATE_Year"] = df["SALEDATE_Year"].fillna(df["sale_date"].dt.year)

        if "SALEDATE_MonthofYearNumber" not in df.columns:
            df["SALEDATE_MonthofYearNumber"] = df["sale_date"].dt.month
        else:
            df["SALEDATE_MonthofYearNumber"] = df["SALEDATE_MonthofYearNumber"].fillna(df["sale_date"].dt.month)

        if "SALEDATE_Quarter" not in df.columns:
            df["SALEDATE_Quarter"] = df["sale_date"].dt.quarter
        else:
            df["SALEDATE_Quarter"] = df["SALEDATE_Quarter"].fillna(df["sale_date"].dt.quarter)

        if "SALEDATE_WeekofYearNumber" not in df.columns:
            try:
                df["SALEDATE_WeekofYearNumber"] = df["sale_date"].dt.isocalendar().week.astype("float")
            except Exception:
                df["SALEDATE_WeekofYearNumber"] = np.nan
        else:
            try:
                df["SALEDATE_WeekofYearNumber"] = df["SALEDATE_WeekofYearNumber"].fillna(
                    df["sale_date"].dt.isocalendar().week.astype("float")
                )
            except Exception:
                pass

    # ------------------------------------------------------------------
    # 2) IsEV cleanup (binary)
    # ------------------------------------------------------------------
    if "IsEV" in df.columns:
        df["IsEV"] = pd.to_numeric(df["IsEV"], errors="coerce").fillna(0).clip(0, 1).astype(np.int8)
    else:
        df["IsEV"] = np.int8(0)

    # ------------------------------------------------------------------
    # 3) Vehicle_condition_overall: extract numeric + missing flag
    # ------------------------------------------------------------------
    if "Vehicle_condition_overall" in df.columns:
        vc = df["Vehicle_condition_overall"].astype(str).str.extract(r"(\d+(\.\d+)?)")[0]
        df["Vehicle_condition_overall"] = pd.to_numeric(vc, errors="coerce")
    else:
        df["Vehicle_condition_overall"] = np.nan

    df["Vehicle_condition_overall_missing"] = df["Vehicle_condition_overall"].isna().astype(np.int8)

    # ------------------------------------------------------------------
    # 4) Fill missing condition using VRLIGHTS (universal smoothed-mean mapping)
    # ------------------------------------------------------------------
    if "VRLIGHTS" in df.columns:
        lights = df["VRLIGHTS"].astype("string").str.strip().str.upper()

        known_mask = df["Vehicle_condition_overall"].notna() & lights.notna()

        global_mean = (
            float(df.loc[df["Vehicle_condition_overall"].notna(), "Vehicle_condition_overall"].mean())
            if df["Vehicle_condition_overall"].notna().any()
            else 3.0
        )

        if known_mask.any():
            stats = (
                pd.DataFrame({
                    "VRLIGHTS": lights.loc[known_mask],
                    "cond": df.loc[known_mask, "Vehicle_condition_overall"]
                })
                .groupby("VRLIGHTS")["cond"]
                .agg(["count", "mean"])
                .rename(columns={"count": "n", "mean": "mu"})
            )
            stats["mu_smooth"] = (stats["n"] * stats["mu"] + k_smooth * global_mean) / (stats["n"] + k_smooth)
            light_map = stats["mu_smooth"].to_dict()
        else:
            light_map = {}

        missing_mask = df["Vehicle_condition_overall"].isna()
        proxy = lights.map(light_map)

        df.loc[missing_mask, "Vehicle_condition_overall"] = proxy.loc[missing_mask]
        df["condition_imputed_from_lights"] = (missing_mask & proxy.notna()).astype(np.int8)

        df["Vehicle_condition_overall"] = df["Vehicle_condition_overall"].fillna(global_mean)

        if drop_vrlights:
            df = df.drop(columns=["VRLIGHTS"], errors="ignore")
    else:
        df["condition_imputed_from_lights"] = np.int8(0)

    # ------------------------------------------------------------------
    # 5) Numeric coercions (core numeric columns)
    # ------------------------------------------------------------------
    num_cols = [
        "VRMILEAGE", "Vehicle_year", "Vehicle_cylinders", "Vehicle_doors",
        "Vehicle_engine", "EngineHP", "BasePrice",
        "SALEDATE_WeekofYearNumber", "SALEDATE_MonthofYearNumber",
        "SALEDATE_Quarter", "SALEDATE_Year"
    ]
    for c in num_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    # Numeric "N/A" handling: sentinel + missing flags
    for c in ["EngineHP", "BasePrice", "Vehicle_cylinders"]:
        if c in df.columns:
            df[f"{c}_missing"] = df[c].isna().astype(np.int8)
            df[c] = df[c].fillna(-1)

    # ------------------------------------------------------------------
    # 6) Clamp year + vehicle_age at sale
    # ------------------------------------------------------------------
    if "Vehicle_year" in df.columns:
        df["Vehicle_year"] = df["Vehicle_year"].clip(1900, 2100)

    if "SALEDATE_Year" in df.columns and "Vehicle_year" in df.columns:
        df["vehicle_age"] = (df["SALEDATE_Year"] - df["Vehicle_year"]).clip(lower=0)
    else:
        df["vehicle_age"] = np.nan

    # ------------------------------------------------------------------
    # 7) Mileage hygiene + safe ratios/logs
    # ------------------------------------------------------------------
    if "VRMILEAGE" in df.columns:
        df["VRMILEAGE"] = df["VRMILEAGE"].clip(lower=0)

        if df["VRMILEAGE"].notna().sum() > 50:
            cap = df["VRMILEAGE"].quantile(0.9995)
            df["VRMILEAGE"] = df["VRMILEAGE"].clip(upper=cap)

        age_safe = df["vehicle_age"].replace(0, 0.5)
        df["mileage_per_year"] = df["VRMILEAGE"] / age_safe
        df["log_mileage"] = np.log1p(df["VRMILEAGE"])
    else:
        df["mileage_per_year"] = np.nan
        df["log_mileage"] = np.nan

    df["log_age"] = np.log1p(df["vehicle_age"])

    # ------------------------------------------------------------------
    # 8) Drivable flag
    # ------------------------------------------------------------------
    if "Vehicle_condition_drivable" in df.columns:
        df["drivable_flag"] = (df["Vehicle_condition_drivable"].astype(str).str.upper() == "Y").astype(np.int8)
    else:
        df["drivable_flag"] = np.int8(0)

    # ------------------------------------------------------------------
    # 9) GVWR numeric class extraction
    # ------------------------------------------------------------------
    if "GVWR" in df.columns:
        gvwr_class = df["GVWR"].astype(str).str.extract(r"Class\s*(\d+)")[0]
        df["GVWR_class"] = pd.to_numeric(gvwr_class, errors="coerce")

    # ------------------------------------------------------------------
    # 10) SMART categorical missing handling (EV-aware)
    # ------------------------------------------------------------------
    unknown_if_missing = [
        "Trim", "Series", "FuelTypePrimary", "DriveType", "BodyClass",
        "MarketSegment", "Vehicle_trantype", "Vehicle_drive"
    ]
    for c in unknown_if_missing:
        if c in df.columns:
            df[c] = df[c].astype("object").fillna("Unknown")

    # EVType: only applicable if IsEV==1
    if "EVType" in df.columns:
        df["EVType"] = df["EVType"].astype("object")
        df.loc[df["IsEV"] == 0, "EVType"] = df.loc[df["IsEV"] == 0, "EVType"].fillna("Not Applicable")
        df.loc[df["IsEV"] == 1, "EVType"] = df.loc[df["IsEV"] == 1, "EVType"].fillna("Unknown")

    # Engine descriptors: often not applicable for EVs; unknown for ICE if missing
    for c in ["EngineConfiguration", "EngineModel"]:
        if c in df.columns:
            df[c] = df[c].astype("object")
            df.loc[df["IsEV"] == 1, c] = df.loc[df["IsEV"] == 1, c].fillna("Not Applicable")
            df.loc[df["IsEV"] == 0, c] = df.loc[df["IsEV"] == 0, c].fillna("Unknown")

    # Process/status fields
    for c in ["Vehicle_condition_crstatus", "Vehicle_condition_drivable"]:
        if c in df.columns:
            df[c] = df[c].astype("object").fillna("Unknown")

    # ------------------------------------------------------------------
    # 11) Downcast numerics (optional)
    # ------------------------------------------------------------------
    for c in df.select_dtypes(include=[np.number]).columns:
        df[c] = pd.to_numeric(df[c], downcast="float")

    # ------------------------------------------------------------------
    # 12) DROP raw date columns to avoid leakage into modeling
    # ------------------------------------------------------------------
    if drop_dates:
        df = df.drop(columns=[c for c in ["VRSALEDATE", "sale_date"] if c in df.columns], errors="ignore")

    return df

In [6]:
def apply_te_maps(df: pd.DataFrame, te_maps: dict, high_card: list) -> pd.DataFrame:
    df = df.copy()
    for col in high_card:
        g = float(te_maps[col]["global_mean"])
        m = te_maps[col]["mapping"]
        out_col = f"te__{col}"

        if col not in df.columns:
            df[out_col] = g
        else:
            s = df[col].astype("object").fillna("__MISSING__")
            df[out_col] = s.map(m).fillna(g)

        df[out_col] = pd.to_numeric(df[out_col], errors="coerce").fillna(g).astype("float32")
    return df

In [7]:
def make_features(df_clean: pd.DataFrame, bundle: dict) -> pd.DataFrame:
    df = apply_te_maps(df_clean, bundle["te_maps"], bundle["high_card"])
    needed = bundle["numeric_feats"] + bundle["categorical_feats"]
    X = df[needed].copy()

    for c in bundle["numeric_feats"]:
        X[c] = pd.to_numeric(X[c], errors="coerce").astype("float32")
    for c in bundle["categorical_feats"]:
        X[c] = X[c].astype("object")
    return X

In [8]:
import numpy as np
from sklearn.base import BaseEstimator, RegressorMixin, clone

class LogTargetRegressor(BaseEstimator, RegressorMixin):
    def __init__(self, base_estimator):
        self.base_estimator = base_estimator

    def fit(self, X, y):
        self.est_ = clone(self.base_estimator)
        y_log = np.log1p(np.asarray(y, dtype=float))
        self.est_.fit(X, y_log)
        return self

    def predict(self, X):
        y_log_pred = self.est_.predict(X)
        return np.expm1(y_log_pred)


In [9]:
# Load (trusted bundle only) :contentReference[oaicite:6]{index=6}
bundle = joblib.load(BUNDLE_PATH)

In [10]:
df_new = pd.read_excel("Test Customer Data.xlsx")
df_new = clean_and_fix_temporal(df_new)  # you must import or include this function

X_new = make_features(df_new, bundle)

In [ ]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

df_new.head(50)

,Location_Region,VRLOCATION,VRMILEAGE,Vehicle_year,Trim,Series,Vehicle_cylinders,Vehicle_botcolor,Vehicle_doors,Vehicle_trantype,Vehicle_engine,Vehicle_drive,Vehicle_condition_drivable,Vehicle_condition_overall,BodyClass,DriveType,EngineHP,FuelTypePrimary,EngineConfiguration,EngineModel,GVWR,PlantCountry,BasePrice,IsEV,EVType,MarketSegment,SALEDATE_WeekofYearNumber,SALEDATE_MonthofYearNumber,SALEDATE_Quarter,SALEDATE_Year,Make,Model,VRVIN,Misc_SalesChannel,Vehicle_condition_overall_missing,condition_imputed_from_lights,EngineHP_missing,BasePrice_missing,Vehicle_cylinders_missing,vehicle_age,mileage_per_year,log_mileage,log_age,drivable_flag,GVWR_class
0,SOUTH,GREENVILLE,226159.0,2011.0,Limited/Touring Convertible,JS,6.0,BLK,4.0,A,3.6,F,N,1.0,Sedan/Saloon,FWD/Front-Wheel Drive,-1.0,Gasoline,Unknown,ERB,"Class 1: 6,000 lb or less (2,722 kg or less)",UNITED STATES (USA),-1.0,0,ICE,Sedan,6.0,2.0,1.0,2025.0,CHRYSLER,200,1C3BC2FG9BN590323,IN LANE,0,0,1,1,0,14.0,16154.214286,12.328998,2.708050,0,1.0
1,WEST,NORTHCALIFORNIA,212703.0,2010.0,SXT,PM,4.0,ORG,4.0,A,2.0,F,S,1.0,Hatchback/Liftback/Notchback,FWD/Front-Wheel Drive,-1.0,Gasoline,Unknown,Unknown,"Class 1: 6,000 lb or less (2,722 kg or less)",UNITED STATES (USA),-1.0,0,ICE,Hatchback,1.0,1.0,1.0,2025.0,DODGE,Caliber,1B3CB4HA0AD570961,IN LANE,0,0,1,1,0,15.0,14180.200000,12.267657,2.772589,0,1.0
2,WEST,NORTHCALIFORNIA,106264.0,2013.0,Limited/RT,Unknown,4.0,GRY,4.0,A,1.4,F,N,1.0,Sedan/Saloon,FWD/Front-Wheel Drive,-1.0,Gasoline,Unknown,Unknown,"Class 1: 6,000 lb or less (2,722 kg or less)",UNITED STATES (USA),-1.0,0,ICE,Sedan,17.0,4.0,2.0,2025.0,DODGE,Dart,1C3CDFCH6DD133483,SIMULCAST,0,0,1,1,0,12.0,8855.333333,11.573691,2.564949,0,1.0
3,WEST,NORTHCALIFORNIA,177982.0,2015.0,SXT,Unknown,4.0,BLK,4.0,A,2.4,F,S,1.0,Sedan/Saloon,FWD/Front-Wheel Drive,-1.0,Gasoline,Unknown,Unknown,"Class 1: 6,000 lb or less (2,722 kg or less)",UNITED STATES (USA),-1.0,0,ICE,Sedan,28.0,7.0,3.0,2025.0,DODGE,Dart,1C3CDFBB7FD182109,SIMULCAST,0,0,1,1,0,10.0,17798.200000,12.089443,2.397895,0,1.0
4,SOUTH,TULSA,238325.0,2009.0,R/T,PM,4.0,SIL,4.0,A,2.4,F,N,1.0,Hatchback/Liftback/Notchback,FWD/Front-Wheel Drive,-1.0,Gasoline,Unknown,Unknown,"Class 1: 6,000 lb or less (2,722 kg or less)",UNITED STATES (USA),-1.0,0,ICE,Hatchback,21.0,5.0,2.0,2025.0,DODGE,Caliber,1B3HB78B09D170796,IN LANE,0,0,1,1,0,16.0,14895.312500,12.381394,2.833213,0,1.0
5,SOUTH,PENSACOLA,200113.0,2012.0,SE,JS,4.0,WHT,4.0,A,2.4,F,Y,1.0,Sedan/Saloon,FWD/Front-Wheel Drive,-1.0,Gasoline,Unknown,Unknown,"Class 1: 6,000 lb or less (2,722 kg or less)",UNITED STATES (USA),-1.0,0,ICE,Sedan,21.0,5.0,2.0,2025.0,DODGE,Avenger,1C3CDZAB0CN170148,IN LANE,0,0,1,1,0,13.0,15393.307692,12.206642,2.639057,1,1.0
6,SOUTH,GREENVILLE,237783.0,2014.0,Sport,Unknown,4.0,GRY,4.0,A,2.0,F,N,1.0,Sport Utility Vehicle (SUV)/Multi-Purpose Vehicle (MPV),FWD/Front-Wheel Drive,-1.0,Gasoline,Unknown,Unknown,"Class 1C: 4,001 - 5,000 lb (1,814 - 2,268 kg)",UNITED STATES (USA),-1.0,0,ICE,SUV,49.0,12.0,4.0,2025.0,JEEP,Patriot,1C4NJPBA8ED789225,IN LANE,0,0,1,1,0,11.0,21616.636364,12.379118,2.484907,0,1.0
7,WEST,NORTHCALIFORNIA,219369.0,2014.0,Sport,Unknown,4.0,SIL,4.0,A,2.0,F,S,1.0,Sport Utility Vehicle (SUV)/Multi-Purpose Vehicle (MPV),FWD/Front-Wheel Drive,-1.0,Gasoline,Unknown,Unknown,"Class 1C: 4,001 - 5,000 lb (1,814 - 2,268 kg)",UNITED STATES (USA),-1.0,0,ICE,SUV,24.0,6.0,2.0,2025.0,JEEP,Patriot,1C4NJPBA5ED502262,IN LANE,0,0,1,1,0,11.0,19942.636364,12.298515,2.484907,0,1.0
8,WEST,NORTHHOUSTON,191947.0,2005.0,BASE,Unknown,6.0,BLU,4.0,A,3.2,F,N,1.0,Sedan/Saloon,Unknown,270.0,Gasoline,V-Shaped,J32A3,"Class 1C: 4,001 - 5,000 lb (1,814 - 2,268 kg)",UNITED STATES (USA),-1.0,0,ICE,Sedan,19.0,5.0,2.0,2025.0,ACURA,TL,19UUA66245A037162,IN LANE,0,0,0,1,0,20.0,9597.350000,12.164980,3.044523,0,1.0
9,WEST,HOUSTON,156039.0,2007.0,Base,Unknown,6.0,BLK,4.0,A,3.2,F,S,1.0,Sedan/Saloon,Unknown,258.0,Gasoline,V-Shaped,J32A3,"Class 1C: 4,001 - 5,000 lb (1,814 - 2,268 kg)",UNITED STATES (USA),-1.0,0,ICE,Sedan

In [12]:
X_new.head()

,VRMILEAGE,Vehicle_year,Vehicle_cylinders,Vehicle_doors,Vehicle_engine,Vehicle_condition_overall,EngineHP,BasePrice,SALEDATE_WeekofYearNumber,SALEDATE_MonthofYearNumber,SALEDATE_Quarter,SALEDATE_Year,vehicle_age,mileage_per_year,log_mileage,log_age,drivable_flag,GVWR_class,Vehicle_condition_overall_missing,condition_imputed_from_lights,EngineHP_missing,BasePrice_missing,Vehicle_cylinders_missing,te__Model,te__Trim,te__Series,te__EngineModel,Location_Region,VRLOCATION,Vehicle_botcolor,Vehicle_trantype,Vehicle_drive,Vehicle_condition_drivable,BodyClass,DriveType,FuelTypePrimary,EngineConfiguration,GVWR,PlantCountry,IsEV,EVType,MarketSegment,Make,Misc_SalesChannel
0,226159.0,2011.0,6.0,4.0,3.6,1.0,-1.0,-1.0,6.0,2.0,1.0,2025.0,14.0,16154.213867,12.328998,2.708050,0.0,1.0,0.0,0.0,1.0,1.0,0.0,4974.971191,5125.483398,2373.515381,4616.384277,SOUTH,GREENVILLE,BLK,A,F,N,Sedan/Saloon,FWD/Front-Wheel Drive,Gasoline,Unknown,"Class 1: 6,000 lb or less (2,722 kg or less)",UNITED STATES (USA),0,ICE,Sedan,CHRYSLER,IN LANE
1,212703.0,2010.0,4.0,4.0,2.0,1.0,-1.0,-1.0,1.0,1.0,1.0,2025.0,15.0,14180.200195,12.267657,2.772589,0.0,1.0,0.0,0.0,1.0,1.0,0.0,1634.803467,5881.673340,1634.803467,8661.553711,WEST,NORTHCALIFORNIA,ORG,A,F,S,Hatchback/Liftback/Notchback,FWD/Front-Wheel Drive,Gasoline,Unknown,"Class 1: 6,000 lb or less (2,722 kg or less)",UNITED STATES (USA),0,ICE,Hatchback,DODGE,IN LANE
2,106264.0,2013.0,4.0,4.0,1.4,1.0,-1.0,-1.0,17.0,4.0,2.0,2025.0,12.0,8855.333008,11.573691,2.564949,0.0,1.0,0.0,0.0,1.0,1.0,0.0,4356.384766,4118.617188,8563.802734,8661.553711,WEST,NORTHCALIFORNIA,GRY,A,F,N,Sedan/Saloon,FWD/Front-Wheel Drive,Gasoline,Unknown,"Class 1: 6,000 lb or less (2,722 kg or less)",UNITED STATES (USA),0,ICE,Sedan,DODGE,SIMULCAST
3,177982.0,2015.0,4.0,4.0,2.4,1.0,-1.0,-1.0,28.0,7.0,3.0,2025.0,10.0,17798.199219,12.089443,2.397895,0.0,1.0,0.0,0.0,1.0,1.0,0.0,4356.384766,5881.673340,8563.802734,8661.553711,WEST,NORTHCALIFORNIA,BLK,A,F,S,Sedan/Saloon,FWD/Front-Wheel Drive,Gasoline,Unknown,"Class 1: 6,000 lb or less (2,722 kg or less)",UNITED STATES (USA),0,ICE,Sedan,DODGE,SIMULCAST
4,238325.0,2009.0,4.0,4.0,2.4,1.0,-1.0,-1.0,21.0,5.0,2.0,2025.0,16.0,14895.312500,12.381394,2.833213,0.0,1.0,0.0,0.0,1.0,1.0,0.0,1634.803467,11262.291016,1634.803467,8661.553711,SOUTH,TULSA,SIL,A,F,N,Hatchback/Liftback/Notchback,FWD/Front-Wheel Drive,Gasoline,Unknown,"Class 1: 6,000 lb or less (2,722 kg or less)",UNITED STATES (USA),0,ICE,Hatchback,DODGE,IN LANE


In [8]:
p_stack = bundle["stack_model"].predict(X_new)
p_lgbm  = bundle["lgbm_model"].predict(X_new)

P = np.column_stack([p_stack, p_lgbm])
p_blend = P @ bundle["blend_w"]
pred    = bundle["iso"].transform(p_blend)

df_new["PredictedSalePrice"] = pred
df_new.to_excel("Scored_New_Customer_Data_Stack.xlsx", index=False)

C:\Users\ChenChen\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
